# 📦 Topic 02: Data & Model Versioning

## 1. Why Version Data and Models?
Git is optimized for text files (code), not multi-gigabyte datasets or binary model weights.
Attempting to store large datasets directly in Git leads to repository bloat and slow clone times.

**Solution:** Data Version Control (DVC) pattern.
- Store actual datasets in remote object storage (S3, GCS, Azure Blob, or local storage).
- Store small pointer/metadata files (`.dvc`) in Git containing dataset MD5 content hashes.

---

## 2. Hands-on: Building a Lightweight Data Versioning System

We will implement a Python simulation of DVC (`MiniDVC`) to hash data, create version pointers, and track data mutations.


In [ ]:
import hashlib
import json
import os
import pandas as pd

class MiniDVC:
    def __init__(self, storage_dir="dvc_remote_storage"):
        self.storage_dir = storage_dir
        os.makedirs(self.storage_dir, exist_ok=True)

    def _compute_md5(self, filepath):
        hasher = hashlib.md5()
        with open(filepath, 'rb') as f:
            buf = f.read()
            hasher.update(buf)
        return hasher.hexdigest()

    def add(self, data_filepath):
        md5_hash = self._compute_md5(data_filepath)
        filename = os.path.basename(data_filepath)
        
        meta_filepath = data_filepath + ".dvc"
        metadata = {
            "path": filename,
            "md5": md5_hash,
            "size_bytes": os.path.getsize(data_filepath)
        }
        with open(meta_filepath, "w") as f:
            json.dump(metadata, f, indent=2)
            
        cas_path = os.path.join(self.storage_dir, f"{md5_hash}_{filename}")
        with open(data_filepath, "rb") as src, open(cas_path, "wb") as dst:
            dst.write(src.read())
            
        print(f"🔒 Tracked '{filename}' -> Hash: {md5_hash[:10]}... | Saved pointer: {meta_filepath}")
        return md5_hash

    def verify_integrity(self, data_filepath):
        meta_filepath = data_filepath + ".dvc"
        if not os.path.exists(meta_filepath):
            print("❌ No pointer metadata found.")
            return False
            
        with open(meta_filepath, "r") as f:
            meta = json.load(f)
            
        current_hash = self._compute_md5(data_filepath)
        if current_hash == meta["md5"]:
            print(f"✅ Data Integrity Intact! ({data_filepath})")
            return True
        else:
            print(f"⚠️ DATA DRIFT / TAMPERING DETECTED! Expected {meta['md5'][:10]}, got {current_hash[:10]}")
            return False

# --- Hands-on Demonstration ---
df_v1 = pd.DataFrame({"feature1": [1.0, 2.5, 3.8], "target": [0, 1, 1]})
df_v1.to_csv("train_dataset.csv", index=False)

dvc = MiniDVC()
hash_v1 = dvc.add("train_dataset.csv")

dvc.verify_integrity("train_dataset.csv")

df_v2 = pd.DataFrame({"feature1": [1.0, 2.5, 3.8, 99.0], "target": [0, 1, 1, 0]})
df_v2.to_csv("train_dataset.csv", index=False)

dvc.verify_integrity("train_dataset.csv")
